# Jam Transformer — Colab 학습 검증 (소량 스텝)

GitHub의 **TRAINING_GUIDE** 흐름을 Colab(Docker 없이 Python 직접)으로 따라가, 실제 전처리 데이터로
**아주 적은 스텝**만 돌려 학습 파이프라인이 정상인지 확인합니다.

| 단계 | 내용 |
|---|---|
| 0 | GPU 확인 (T4) |
| 1 | 저장소 클론 (`feat/single-stream-accompaniment`) |
| 2 | v3.1.0 릴리즈에서 전처리 데이터(18,161 shard) 다운로드 |
| 3 | 의존성 설치 (torch는 Colab 내장) |
| 4 | `--fast_dev_run` (1 스텝: 배선·fingerprint·차원 검증) |
| 5 | `--dry_run_steps 20` (초기 loss·peak VRAM·ms/step) |

> **런타임**: `런타임 > 런타임 유형 변경 > T4 GPU`.
> **batch_size=8**: T4는 Flash Attention 미지원(SM 7.5)이라 메모리가 빡빡 → 안전한 소량 검증용.
> (빌린 4070 Ti Super(16GB, Ampere)에서는 기본 batch=32 그대로 가능.)


## 0. GPU 확인

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU (런타임을 T4로 변경)')

## 1. 저장소 클론

In [ ]:
%cd /content
!rm -rf symbolic-accompaniment-transformer
!git clone -q -b feat/single-stream-accompaniment https://github.com/stemkwk/symbolic-accompaniment-transformer.git
%cd symbolic-accompaniment-transformer

## 2. v3.1.0 릴리즈에서 전처리 데이터 다운로드 + 압축 해제

In [ ]:
!wget -q https://github.com/stemkwk/symbolic-accompaniment-transformer/releases/download/v3.1.0/jam_data_processed.zip
!unzip -q jam_data_processed.zip -d .
import json, collections
d = json.load(open('data/processed/_chunk_index.json'))
print('shards:', dict(collections.Counter(k.split('_')[0] for k in d)), '| total', len(d))

## 3. 의존성 설치 (torch는 Colab 내장 → 재설치 안 함)

In [ ]:
!pip install -q pytorch-lightning miditoolkit pretty_midi pyyaml loguru python-dotenv
!pip install -q -e . --no-build-isolation 2>&1 | tail -2
print('설치 완료')

## 4. ① fast_dev_run — 1 스텝 검증

에러 없이 `Epoch 0: 1/1`이 뜨면 → **데이터-config fingerprint 일치**(`assert_data_matches_config`)
+ 모델 배선·텐서 차원 정상.


In [ ]:
import os
os.environ['WANDB_DISABLED'] = 'true'
!python scripts/train.py --data_dir data/processed --fast_dev_run \
  --set training.batch_size=8 \
  --set training.log_to_file=false --set training.csv_logger_enabled=false

## 5. ② dry_run 20 스텝 — 초기 loss / VRAM / 속도

확인 포인트:
- **초기 loss ≈ ln(173) ≈ 5.15** (4.9~5.4 정상 / 8↑이면 마스킹·임베딩 버그 의심)
- **peak VRAM** (T4 16GB 대비 여유)
- **ms/step** → epoch·전체 학습 시간 추정


In [ ]:
!python scripts/train.py --data_dir data/processed --dry_run_steps 20 \
  --set training.batch_size=8

---
## 정상 판정 기준 요약
- Cell 2: `{'pop909': 909, 'lakh': 15897, 'slakh': 1355} | total 18161`
- Cell 4: 에러 없이 1 스텝 완료
- Cell 5: 초기 loss 4.9~5.4, OOM 없음

> 빌린 박스(16GB Ampere)에서는 `--set training.batch_size=8`을 빼고 기본 32로 돌려 실제 VRAM/속도를 측정하세요.
